# JWST MIRI LRS Spectroscopic Reduction of K2-18b
## A Complete Eureka! Pipeline Tutorial - Stages 1 through 6

---

**Author:** Chetan Prakash 
**Reference:** Tutorial based on the [Eureka! pipeline](https://eurekadocs.readthedocs.io) (Bell et al. 2022)  
**Target:** K2-18b - a sub-Neptune in the habitable zone of an M2.5V star  
**Instrument:** JWST MIRI Low-Resolution Spectrometer (LRS), Slitless mode  
**Science Case:** Transmission spectroscopy (5–12 μm) to characterise the atmosphere of K2-18b  

---

## Scientific Background

K2-18b was discovered by the K2 mission (Montet et al. 2015) and is one of the most studied sub-Neptunes (~2.7 R⊕, ~8.6 M⊕) sitting squarely in the habitable zone of its M-dwarf host. Madhusudhan et al. (2023) reported tentative evidence for CH₄, CO₂, and possibly dimethyl sulfide (DMS), making it a prime target for MIRI LRS follow-up in the 5–12 μm range where CO₂ has a strong band at 15 μm, and where NH₃, SO₂, and other biosignature candidates have diagnostic features.

MIRI LRS slitless mode delivers R ~ 40–160 across 5–12 μm and is optimised for bright targets observed in FASTR1 readout mode.

---

## K2-18 System Parameters

| Parameter | Value | Reference |
|---|---|---|
| R★ (R☉) | 0.4445 | Cloutier et al. 2017 |
| M★ (M☉) | 0.3593 | Cloutier et al. 2017 |
| T_eff (K) | 3457 | Sarkis et al. 2018 |
| log g | 4.96 | Sarkis et al. 2018 |
| [Fe/H] | 0.12 | Sarkis et al. 2018 |
| Period (d) | 32.940038 | Benneke et al. 2019 |
| T₀ (BJD_TDB) | 2457264.39116 | Benneke et al. 2019 |
| Rp/R★ | 0.05875 | Madhusudhan et al. 2023 |
| a/R★ | 72.77 | Madhusudhan et al. 2023 |
| Inclination (°) | 89.5785 | Madhusudhan et al. 2023 |
| e | 0 (fixed) | — |
| T_dur (h) | 2.858 | Benneke et al. 2019 |

---

## Pipeline Overview

```
MAST uncal files  ──►  Stage 1  ──►  Stage 2  ──►  Stage 3
  (detector ramps)    (cal.fits)    (calints)    (x1dints + 2D spectra)
                                                      │
                                          Stage 4 ◄───┘
                                        (light curves per λ-bin)
                                              │
                                          Stage 5
                                        (MCMC/nested sampling fits)
                                              │
                                          Stage 6
                                        (transmission spectrum + plots)
```

---

## Prerequisites

```bash
conda create -n eureka python=3.10
conda activate eureka
pip install eureka[all]
# Ensure CRDS_PATH and CRDS_SERVER_URL are set:
export CRDS_PATH=$HOME/crds_cache
export CRDS_SERVER_URL=https://jwst-crds.stsci.edu
```


## Table of Contents

1. [Environment Setup & Data Download](#1-environment-setup--data-download)
2. [Stage 1 — Detector-Level Corrections](#2-stage-1--detector-level-corrections)
3. [Stage 2 — Spectroscopic Pipeline](#3-stage-2--spectroscopic-pipeline)
4. [Stage 3 — 1D Spectral Extraction](#4-stage-3--1d-spectral-extraction)
5. [Stage 4 — Light Curve Construction](#5-stage-4--light-curve-construction)
6. [Stage 5 — Light Curve Fitting](#6-stage-5--light-curve-fitting)
7. [Stage 6 — Transmission Spectrum & Visualisation](#7-stage-6--transmission-spectrum--visualisation)
8. [Interpreting Your Results](#8-interpreting-your-results)


---
## 1. Environment Setup & Data Download


### 1.1 CRDS Configuration

The JWST Calibration Reference Data System (CRDS) provides the reference files used by Stages 1 and 2. Eureka! calls the official `jwst` pipeline under the hood, so CRDS must be reachable.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CRDS environment variables — set these BEFORE importing jwst pipeline modules
# ─────────────────────────────────────────────────────────────────────────────
os.environ['CRDS_PATH']       = os.path.expanduser('~/crds_cache')
os.environ['CRDS_SERVER_URL'] = 'https://jwst-crds.stsci.edu'

# The pipeline context pins which set of reference files to use.
os.environ['CRDS_CONTEXT'] = 'jwst_1464.pmap'   # latest currnetly. Can set CRDS_CONTEXT=jwst_latest but setting specific is better for reproducibility. 

# Verify
print("CRDS_PATH       :", os.environ['CRDS_PATH'])
print("CRDS_SERVER_URL :", os.environ['CRDS_SERVER_URL'])
print("CRDS_CONTEXT    :", os.environ.get('CRDS_CONTEXT', 'default (latest)'))


### 1.2 Downloading K2-18b Data from MAST

K2-18b MIRI LRS observations were obtained under GO programme **2722** (PI: Madhusudhan). The raw (uncalibrated) files end in `_uncal.fits` and are what Stage 1 expects.

> **Note:** You can also download via the [MAST Portal](https://mast.stsci.edu) or by running the bash code below (.sh also given in the repo).
>
> Best way to do the Eureka! reduction is to create a top dir, eureka inside it, k2-18b, inside it - uncal, stage1, stage2, stage3, stage4, stage5, stage6. Put the downloaded `_uncal.fits` in the uncal folder. Run this notebook in the k2-18b dir.

In [ ]:
import os

# Base directory (home)
home_dir = os.path.expanduser("~")

# Main paths
eureka_dir = os.path.join(home_dir, "eureka")
target_dir = os.path.join(eureka_dir, "k2-18b")

# Subdirectories
subdirs = [
    "uncal",
    "stage1",
    "stage2",
    "stage3",
    "stage4",
    "stage5",
    "stage6"
]

# Create directories
os.makedirs(target_dir, exist_ok=True)

for sub in subdirs:
    os.makedirs(os.path.join(target_dir, sub), exist_ok=True)


In [ ]:
#!/bin/bash
#
# Requires bash version >= 4
# 
# The script uses the command line tool 'curl' for querying
# the MAST Download service for public and protected data. 
#

type -P curl >&/dev/null || { echo "This script requires curl. Exiting." >&2; exit 1; }



# Check for existing Download Folder
# prompt user for overwrite, if found
let EXTENSION=0
FOLDER=MAST_2026-03-31T1305
DOWNLOAD_FOLDER=$FOLDER
if [ -d $DOWNLOAD_FOLDER ]
then
  echo -n "Download folder ${DOWNLOAD_FOLDER} found, (C)ontinue writing to existing folder or use (N)ew folder? [N]> "
  read -n1 ans
  if [ "$ans" = "c" -o "$ans" = "C" ]
  then
    echo ""
    echo "Downloading to existing folder: ${DOWNLOAD_FOLDER}"
    CONT="-C -"
  else
    while [ -d $DOWNLOAD_FOLDER ]
    do
      ((EXTENSION++))
      DOWNLOAD_FOLDER="${FOLDER}-${EXTENSION}"
    done

    echo ""
    echo "Downloading to new folder: ${DOWNLOAD_FOLDER}"
  fi
fi

# mkdir if it doesn't exist and download files there. 
mkdir -p ${DOWNLOAD_FOLDER}

cat >${DOWNLOAD_FOLDER}/MANIFEST.HTML<<EOT
<!DOCTYPE html>
<html>
    <head>
        <title>MAST_2026-03-31T1305</title>
    </head>
    <body>
        <h2>Manifest for File: MAST_2026-03-31T1305</h2>
        <h3>Total Files: 9</h3>
        <table cellspacing="0" cellpadding="4" rules="all" style="border-width:5px; border-style:solid; border-collapse:collapse;">
            <tr>
                <td><b>URI</b></td>
                <td><b>File</b></td>
                <td><b>Access</b></td>
                <td><b>Status</b></td>
                <td><b>Logged In User</b></td>
            </tr>
            
            <tr>
                <td>mast:JWST/product/jw02722002001_04102_00001-seg001_mirimage_uncal.fits</td>
                <td>JWST/jw02722002001_04102_00001-seg001_mirimage/jw02722002001_04102_00001-seg001_mirimage_uncal.fits</td>
                <td>PUBLIC</td>
                <td>OK</td>
                <td>anonymous</td>
            </tr>
            
            <tr>
                <td>mast:JWST/product/jw02722002001_04103_00001-seg005_mirimage_uncal.fits</td>
                <td>JWST/jw02722002001_04103_00001-seg005_mirimage/jw02722002001_04103_00001-seg005_mirimage_uncal.fits</td>
                <td>PUBLIC</td>
                <td>OK</td>
                <td>anonymous</td>
            </tr>
            
            <tr>
                <td>mast:JWST/product/jw02722002001_02101_00001-seg001_mirimage_uncal.fits</td>
                <td>JWST/jw02722002001_02101_00001-seg001_mirimage/jw02722002001_02101_00001-seg001_mirimage_uncal.fits</td>
                <td>PUBLIC</td>
                <td>OK</td>
                <td>anonymous</td>
            </tr>
            
            <tr>
                <td>mast:JWST/product/jw02722002001_04103_00001-seg006_mirimage_uncal.fits</td>
                <td>JWST/jw02722002001_04103_00001-seg006_mirimage/jw02722002001_04103_00001-seg006_mirimage_uncal.fits</td>
                <td>PUBLIC</td>
                <td>OK</td>
                <td>anonymous</td>
            </tr>
            
            <tr>
                <td>mast:JWST/product/jw02722002001_04103_00001-seg003_mirimage_uncal.fits</td>
                <td>JWST/jw02722002001_04103_00001-seg003_mirimage/jw02722002001_04103_00001-seg003_mirimage_uncal.fits</td>
                <td>PUBLIC</td>
                <td>OK</td>
                <td>anonymous</td>
            </tr>
            
            <tr>
                <td>mast:JWST/product/jw02722002001_04103_00001-seg002_mirimage_uncal.fits</td>
                <td>JWST/jw02722002001_04103_00001-seg002_mirimage/jw02722002001_04103_00001-seg002_mirimage_uncal.fits</td>
                <td>PUBLIC</td>
                <td>OK</td>
                <td>anonymous</td>
            </tr>
            
            <tr>
                <td>mast:JWST/product/jw02722002001_04103_00001-seg004_mirimage_uncal.fits</td>
                <td>JWST/jw02722002001_04103_00001-seg004_mirimage/jw02722002001_04103_00001-seg004_mirimage_uncal.fits</td>
                <td>PUBLIC</td>
                <td>OK</td>
                <td>anonymous</td>
            </tr>
            
            <tr>
                <td>mast:JWST/product/jw02722002001_04103_00001-seg001_mirimage_uncal.fits</td>
                <td>JWST/jw02722002001_04103_00001-seg001_mirimage/jw02722002001_04103_00001-seg001_mirimage_uncal.fits</td>
                <td>PUBLIC</td>
                <td>OK</td>
                <td>anonymous</td>
            </tr>
            
            <tr>
                <td>mast:JWST/product/jw02722002001_04103_00001-seg007_mirimage_uncal.fits</td>
                <td>JWST/jw02722002001_04103_00001-seg007_mirimage/jw02722002001_04103_00001-seg007_mirimage_uncal.fits</td>
                <td>PUBLIC</td>
                <td>OK</td>
                <td>anonymous</td>
            </tr>
            
        </table>
    </body>
</html>

EOT

# Download Product Files:



cat <<EOT
<<< Downloading File: mast:JWST/product/jw02722002001_04102_00001-seg001_mirimage_uncal.fits
                  To: ${DOWNLOAD_FOLDER}/JWST/jw02722002001_04102_00001-seg001_mirimage/jw02722002001_04102_00001-seg001_mirimage_uncal.fits
EOT

curl --globoff --location-trusted -f --progress-bar --create-dirs $CONT --output ./${DOWNLOAD_FOLDER}'/JWST/jw02722002001_04102_00001-seg001_mirimage/jw02722002001_04102_00001-seg001_mirimage_uncal.fits' 'https://mast.stsci.edu/api/v0.1/Download/file?bundle_name=MAST_2026-03-31T1305.sh&uri=mast:JWST/product/jw02722002001_04102_00001-seg001_mirimage_uncal.fits'





cat <<EOT
<<< Downloading File: mast:JWST/product/jw02722002001_04103_00001-seg005_mirimage_uncal.fits
                  To: ${DOWNLOAD_FOLDER}/JWST/jw02722002001_04103_00001-seg005_mirimage/jw02722002001_04103_00001-seg005_mirimage_uncal.fits
EOT

curl --globoff --location-trusted -f --progress-bar --create-dirs $CONT --output ./${DOWNLOAD_FOLDER}'/JWST/jw02722002001_04103_00001-seg005_mirimage/jw02722002001_04103_00001-seg005_mirimage_uncal.fits' 'https://mast.stsci.edu/api/v0.1/Download/file?bundle_name=MAST_2026-03-31T1305.sh&uri=mast:JWST/product/jw02722002001_04103_00001-seg005_mirimage_uncal.fits'





cat <<EOT
<<< Downloading File: mast:JWST/product/jw02722002001_02101_00001-seg001_mirimage_uncal.fits
                  To: ${DOWNLOAD_FOLDER}/JWST/jw02722002001_02101_00001-seg001_mirimage/jw02722002001_02101_00001-seg001_mirimage_uncal.fits
EOT

curl --globoff --location-trusted -f --progress-bar --create-dirs $CONT --output ./${DOWNLOAD_FOLDER}'/JWST/jw02722002001_02101_00001-seg001_mirimage/jw02722002001_02101_00001-seg001_mirimage_uncal.fits' 'https://mast.stsci.edu/api/v0.1/Download/file?bundle_name=MAST_2026-03-31T1305.sh&uri=mast:JWST/product/jw02722002001_02101_00001-seg001_mirimage_uncal.fits'





cat <<EOT
<<< Downloading File: mast:JWST/product/jw02722002001_04103_00001-seg006_mirimage_uncal.fits
                  To: ${DOWNLOAD_FOLDER}/JWST/jw02722002001_04103_00001-seg006_mirimage/jw02722002001_04103_00001-seg006_mirimage_uncal.fits
EOT

curl --globoff --location-trusted -f --progress-bar --create-dirs $CONT --output ./${DOWNLOAD_FOLDER}'/JWST/jw02722002001_04103_00001-seg006_mirimage/jw02722002001_04103_00001-seg006_mirimage_uncal.fits' 'https://mast.stsci.edu/api/v0.1/Download/file?bundle_name=MAST_2026-03-31T1305.sh&uri=mast:JWST/product/jw02722002001_04103_00001-seg006_mirimage_uncal.fits'





cat <<EOT
<<< Downloading File: mast:JWST/product/jw02722002001_04103_00001-seg003_mirimage_uncal.fits
                  To: ${DOWNLOAD_FOLDER}/JWST/jw02722002001_04103_00001-seg003_mirimage/jw02722002001_04103_00001-seg003_mirimage_uncal.fits
EOT

curl --globoff --location-trusted -f --progress-bar --create-dirs $CONT --output ./${DOWNLOAD_FOLDER}'/JWST/jw02722002001_04103_00001-seg003_mirimage/jw02722002001_04103_00001-seg003_mirimage_uncal.fits' 'https://mast.stsci.edu/api/v0.1/Download/file?bundle_name=MAST_2026-03-31T1305.sh&uri=mast:JWST/product/jw02722002001_04103_00001-seg003_mirimage_uncal.fits'





cat <<EOT
<<< Downloading File: mast:JWST/product/jw02722002001_04103_00001-seg002_mirimage_uncal.fits
                  To: ${DOWNLOAD_FOLDER}/JWST/jw02722002001_04103_00001-seg002_mirimage/jw02722002001_04103_00001-seg002_mirimage_uncal.fits
EOT

curl --globoff --location-trusted -f --progress-bar --create-dirs $CONT --output ./${DOWNLOAD_FOLDER}'/JWST/jw02722002001_04103_00001-seg002_mirimage/jw02722002001_04103_00001-seg002_mirimage_uncal.fits' 'https://mast.stsci.edu/api/v0.1/Download/file?bundle_name=MAST_2026-03-31T1305.sh&uri=mast:JWST/product/jw02722002001_04103_00001-seg002_mirimage_uncal.fits'





cat <<EOT
<<< Downloading File: mast:JWST/product/jw02722002001_04103_00001-seg004_mirimage_uncal.fits
                  To: ${DOWNLOAD_FOLDER}/JWST/jw02722002001_04103_00001-seg004_mirimage/jw02722002001_04103_00001-seg004_mirimage_uncal.fits
EOT

curl --globoff --location-trusted -f --progress-bar --create-dirs $CONT --output ./${DOWNLOAD_FOLDER}'/JWST/jw02722002001_04103_00001-seg004_mirimage/jw02722002001_04103_00001-seg004_mirimage_uncal.fits' 'https://mast.stsci.edu/api/v0.1/Download/file?bundle_name=MAST_2026-03-31T1305.sh&uri=mast:JWST/product/jw02722002001_04103_00001-seg004_mirimage_uncal.fits'





cat <<EOT
<<< Downloading File: mast:JWST/product/jw02722002001_04103_00001-seg001_mirimage_uncal.fits
                  To: ${DOWNLOAD_FOLDER}/JWST/jw02722002001_04103_00001-seg001_mirimage/jw02722002001_04103_00001-seg001_mirimage_uncal.fits
EOT

curl --globoff --location-trusted -f --progress-bar --create-dirs $CONT --output ./${DOWNLOAD_FOLDER}'/JWST/jw02722002001_04103_00001-seg001_mirimage/jw02722002001_04103_00001-seg001_mirimage_uncal.fits' 'https://mast.stsci.edu/api/v0.1/Download/file?bundle_name=MAST_2026-03-31T1305.sh&uri=mast:JWST/product/jw02722002001_04103_00001-seg001_mirimage_uncal.fits'





cat <<EOT
<<< Downloading File: mast:JWST/product/jw02722002001_04103_00001-seg007_mirimage_uncal.fits
                  To: ${DOWNLOAD_FOLDER}/JWST/jw02722002001_04103_00001-seg007_mirimage/jw02722002001_04103_00001-seg007_mirimage_uncal.fits
EOT

curl --globoff --location-trusted -f --progress-bar --create-dirs $CONT --output ./${DOWNLOAD_FOLDER}'/JWST/jw02722002001_04103_00001-seg007_mirimage/jw02722002001_04103_00001-seg007_mirimage_uncal.fits' 'https://mast.stsci.edu/api/v0.1/Download/file?bundle_name=MAST_2026-03-31T1305.sh&uri=mast:JWST/product/jw02722002001_04103_00001-seg007_mirimage_uncal.fits'




exit 0

### 1.3 Inspect a Raw File

In [ ]:
from astropy.io import fits
import glob
import os
uncal_dir = '/home/chetan/eureka/k2-18b/uncal'
uncal_files = sorted(glob.glob(os.path.join(uncal_dir, '*uncal.fits')))
# ─── Open the first uncal file and print header + array shape ─────────────────
if uncal_files:
    with fits.open(uncal_files[0]) as hdul:
        hdul.info()
        pri_hdr  = hdul[0].header
        sci_data = hdul['SCI'].data   # shape: (nints, ngroups, nrows, ncols)
    print("\n─── Primary Header (selected keys) ───")
    for k in ['TARGNAME','INSTRUME','DETECTOR','FILTER','SUBARRAY',
'NINTS','NGROUPS','NFRAMES','TFRAME','TGROUP','EFFINTTM']:
        print(f"  {k:15s} = {pri_hdr.get(k,'N/A')}")
    print(f"\nSCI array shape (nints, ngroups, rows, cols): {sci_data.shape}")
    print(f"Estimated integration time : {pri_hdr.get('EFFINTTM','?')} s")
    total_hours = sci_data.shape[0] * pri_hdr.get('EFFINTTM', 47) / 3600
    print(f"Total baseline             : ~{total_hours:.2f} h")
else:
    print(f"No uncalibrated FITS files found in {uncal_dir}.")

for f in uncal_files:
    with fits.open(f) as hdul:
        h = hdul[0].header
        print(f"{os.path.basename(f):60s}  FILTER={h.get('FILTER','?'):8s}  NINTS={h.get('NINTS','?')}")

# You can exclude the F1500W files from the uncal folder.       

### 1.4 run_eureka.py

This code will run all our stage 1 to 6 .efc (eureka control file)and .epf (eureka parameter file).

In [ ]:
import os
import eureka.lib.plots
import eureka.S1_detector_processing.s1_process as s1
import eureka.S2_calibrations.s2_calibrate as s2
import eureka.S3_data_reduction.s3_reduce as s3
import eureka.S4_generate_lightcurves.s4_genLC as s4
import eureka.S5_lightcurve_fitting.s5_fit as s5
import eureka.S6_planet_spectra.s6_spectra as s6

eventlabel = ''k2-18b''

ecf_path = '.'+os.sep

if __name__ == '__main__':
    # To skip one or more stages that were already run,
    # just comment them out below

    #meta = s1.rampfitJWST(eventlabel, ecf_path=ecf_path)

    #meta = s2.calibrateJWST(eventlabel, ecf_path=ecf_path)

    #spec, meta = s3.reduce(eventlabel, ecf_path=ecf_path)

    #spec, lc, meta = s4.genlc(eventlabel, ecf_path=ecf_path)

    #meta = s5.fitlc(eventlabel, ecf_path=ecf_path)

    #meta, lc = s6.plot_spectra(eventlabel, ecf_path=ecf_path)

---
## 2. Stage 1 — Detector-Level Corrections

Stage 1 wraps the official JWST pipeline `Detector1Pipeline` and converts raw ramp data (`_uncal.fits`) into rate images (`_rate.fits` and per-integration `_rateints.fits`). The main steps are:

| Step | Description |
|---|---|
| `suffix` | Data file suffix (e.g. uncal) |
| `maximum_cores` | Fraction of processor cores to use when computing the jump step and the ramp fits |
| `jump_rejection_threshold` | A floating-point value that sets the sigma threshold for jump detection. The default is 4.0, but it is often best to increase this number for time-series observations to avoid excessively high false-positives. The optimal value will vary between different datasets and different instruments, but from experience we have found that values around 6.0–8.0 are often reasonable |
| `group_scale` | Rescale groups to account for integration non-linearities |
| `dq_init` | Populate data quality flags from bad pixel mask |
| `saturation` | Flag saturated pixels |
| `firstframe` | Flag the first group (anomalously high) |
| `lastframe` | Flag the last group if truncated |
| `reset` | Subtract the MIRI reset anomaly |
| `linearity` | Apply non-linearity correction |
| `dark_current` | Subtract dark current |
| `refpix` | Correct using reference pixels |
| `jump` | Detect cosmic ray hits |
| `ramp_fitting` | Fit slopes to ramps → rate (DN/s) |


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Write the Stage 1 Eureka Parameter File (S1.ecf) for K2-18b MIRI LRS
# ─────────────────────────────────────────────────────────────────────────────
TOP_DIR = Path('/home/chetan/eureka/k2-18b')
ecf_dir = TOP_DIR
ecf_dir.mkdir(parents=True, exist_ok=True)

s1_ecf_content = """\
# Eureka! Control File for Stage 1: Detector Processing

# Stage 1 Documentation: https://eurekadocs.readthedocs.io/en/latest/ecf.html#stage-1

suffix              uncal

maximum_cores       'half'  # Options are 'none', quarter', 'half','all'

# Pipeline stages
skip_emicorr        False   # Skipped by default despite what the jwst readthedocs page says
skip_saturation     False
skip_firstframe     False   # Typically best to set False for MIRI TSO, but you should experiment
skip_lastframe      False   # Typically best to set False for MIRI TSO, but you should experiment
skip_refpix         False
skip_linearity      False
skip_rscd           False   # Skipped by default for MIRI TSO
skip_dark_current   False
skip_jump           False
skip_ramp_fitting   False

#Pipeline stages parameters
jump_rejection_threshold  15.0 # float, default is 4.0, CR sigma rejection threshold. Usually recommend a larger value for TSO data.

# Custom linearity reference file
custom_linearity    False
linearity_file      /path/to/custom/linearity/fits/file

# Saturation
update_sat_flags    False   # Wheter to update the saturation flags more aggressively
expand_prev_group   False   # Expand saturation flags to previous group
dq_sat_mode         percentile # Options: [percentile, min, defined]
dq_sat_percentile   50      # Percentile of the entire time series to use to define the saturation mask (50=median)
dq_sat_columns      [[0, 0], [0,0], [0,0], [0,0], [0,0]]  #for dq_sat_mode = defined, user defined saturated columns

# Background subtraction
remove_390hz        False    # Use custom Eureka! code to remove the 390 Hz periodic noise in MIRI/LRS SLITLESSPRISM group-level data
grouplevel_bg       False
ncpu                8
bg_y1               6
bg_y2               26
bg_deg              0
bg_method           median  # Options: std (Standard Deviation), median (Median Absolute Deviation), mean (Mean Absolute Deviation)
p3thresh            5

# Diagnostics
isplots_S1          1
nplots              5
hide_plots          True
verbose             True

# Project directory
topdir              /home/chetan/eureka/k2-18b/

# Directories relative to topdir
inputdir		    uncal	
outputdir           stage1
"""

s1_path = ecf_dir / 'S1_k2-18b.ecf'
s1_path.write_text(s1_ecf_content)
print("S1_k2-18b.ecf written to:", s1_path)
print(s1_ecf_content)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Run Stage 1
# ─────────────────────────────────────────────────────────────────────────────
# The output _rateints.fits files (one per integration) go to Stage1/.

eventlabel = 'k2-18b'

ecf_path = '.'+os.sep

if __name__ == '__main__':
    # To skip one or more stages that were already run,
    # just comment them out below

    meta = s1.rampfitJWST(eventlabel, ecf_path=ecf_path)

    #meta = s2.calibrateJWST(eventlabel, ecf_path=ecf_path)

    #spec, meta = s3.reduce(eventlabel, ecf_path=ecf_path)

    #spec, lc, meta = s4.genlc(eventlabel, ecf_path=ecf_path)

    #meta = s5.fitlc(eventlabel, ecf_path=ecf_path)

    #meta, lc = s6.plot_spectra(eventlabel, ecf_path=ecf_path)

---
## 3. Stage 2 — Spectroscopic Pipeline

Stage 2 wraps `Spec2Pipeline` and applies instrument-level calibrations to the rate images:

| Step | Description |
|---|---|
| `flat_field` | Divide by the pixel flat (for LRS slitless: uses `miri_lrs_flat`) |
| `photom` | Apply photometric calibration (flux density per DN/s) |
| `extract_1d` | Extract 1D spectrum (we skip this; Stage 3 does its own extraction) |


In [ ]:
s2_ecf_content = """\
# Eureka! Control File for Stage 2: Data Reduction

# Stage 2 Documentation: https://eurekadocs.readthedocs.io/en/latest/ecf.html#stage-2

suffix                  rateints    # Data file suffix

# Note: different instruments and modes will use different steps by default
skip_flat_field         True    # Set this to False if you want to compute calibrated stellar spectra.
skip_photom             True    # Recommended to skip to get better uncertainties out of Stage 3.
skip_extract_1d         True

# Diagnostics
hide_plots              True    # If True, plots will automatically be closed rather than popping up

# Project directory
topdir                  /home/chetan/eureka/k2-18b/

# Directories relative to topdir
inputdir		        stage1
outputdir               stage2
"""

s2_path = ecf_dir / 'S2_k2-18b.ecf'
s2_path.write_text(s2_ecf_content)
print("S2_k2-18b.ecf written to:", s2_path)
print(s2_ecf_content)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Run Stage 2
# ─────────────────────────────────────────────────────────────────────────────
# The output _calints.fits files go to Stage2/.

eventlabel = 'k2-18b'

ecf_path = '.'+os.sep

if __name__ == '__main__':
    # To skip one or more stages that were already run,
    # just comment them out below

    #meta = s1.rampfitJWST(eventlabel, ecf_path=ecf_path)

    meta = s2.calibrateJWST(eventlabel, ecf_path=ecf_path)

    #spec, meta = s3.reduce(eventlabel, ecf_path=ecf_path)

    #spec, lc, meta = s4.genlc(eventlabel, ecf_path=ecf_path)

    #meta = s5.fitlc(eventlabel, ecf_path=ecf_path)

    #meta, lc = s6.plot_spectra(eventlabel, ecf_path=ecf_path)

---
## 4. Stage 3 — 1D Spectral Extraction

This is the most scientifically critical stage for transit spectroscopy. Eureka! Stage 3 takes the 2D calibrated cubes and produces per-integration 1D spectra. The main operations are:

1. **Background subtraction** — Fit and subtract a polynomial to columns away from the trace
2. **Sigma-clipping** — Remove outliers from each integration's 2D frame
3. **Optimal extraction (Horne 1986)** — Weight the cross-dispersion profile by the signal-to-noise ratio to maximise extraction precision
4. **Wavelength calibration** — Map columns to wavelengths using the WCS from Stage 2

### Key parameters for K2-18b MIRI LRS

- **`bg_hw`**: Half-width of the background region. For MIRI LRS slitless, set to ~10 rows away from the trace.
- **`spec_hw`**: Half-width of the extraction aperture (cross-dispersion). Typically 4–6 pixels for MIRI LRS.
- **`src_pos_type`**: How to find the trace centre. Use `'hst'` (cross-correlation) or `'gaussian'` for MIRI.
- **`record_ypos`**: Save the trace centroid per integration — useful for decorrelation in Stage 5.

In [ ]:
s3_ecf_content = """\
# Eureka! Control File for Stage 3: Data Reduction

# Stage 3 Documentation: https://eurekadocs.readthedocs.io/en/latest/ecf.html#stage-3

ncpu            16          # Number of CPUs
nfiles          8           # The number of data files to analyze simultaneously
max_memory      0.7         # The maximum fraction of memory you want utilized by read-in frames (this will reduce nfiles if need be)
indep_batches   False       # Independently treat each batch of files? Strongly recommended to leave this as False unless you have a clear reason to set it to True.
suffix          calints     # Data file suffix

calibrated_spectra  False   # Set True to generate flux-calibrated spectra/photometry in mJy
                            # Set False to convert to electrons

# Subarray region of interest
ywindow         [10, 386]   # Vertical axis as seen in DS9
xwindow         [16, 61]    # Horizontal axis as seen in DS9
src_pos_type    gaussian    # Determine source position when not given in header (Options: header, gaussian, weighted, max, hst, or a numeric value)
record_ypos     True        # Option to record the y position and width for each integration (only records if src_pos_type is gaussian)
dqmask          True        # Mask pixels with an odd entry in the DQ array
expand          1           # Super-sampling factor along cross-dispersion direction

# Outlier rejection along time axis
ff_outlier      True        # Set False to use only background region (recommended for deep transits)
                            # Set True to use full frame (works well for shallow transits/eclipses)
bg_thresh       [3,3]       # Double-iteration X-sigma threshold for outlier rejection along time axis

# Background parameters
bg_hw           10          # Half-width of exclusion region for BG subtraction (relative to source position)
bg_deg          0           # Polynomial order for column-by-column background subtraction, -1 for median of entire frame
bg_method       mean        # Options: std (Standard Deviation), median (Median Absolute Deviation), mean (Mean Absolute Deviation)
p3thresh        7           # X-sigma threshold for outlier rejection during background subtraction

# Spectral extraction parameters
spec_hw         4           # Half-width of aperture region for spectral extraction (relative to source position)
fittype         meddata     # Method for constructing spatial profile (Options: smooth, meddata, poly, gauss, wavelet, or wavelet2D)
median_thresh   5           # Sigma threshold when flagging outliers in median frame, when fittype=meddata and window_len > 1
window_len      15          # Smoothing window length, for median frame or when fittype = smooth or meddata (when computing median frame). Can set to 1 for no smoothing when computing median frame for fittype=meddata.
prof_deg        3           # Polynomial degree, when fittype = poly
p5thresh        7           # X-sigma threshold for outlier rejection while constructing spatial profile
p7thresh        7           # X-sigma threshold for outlier rejection during optimal spectral extraction

# Diagnostics
isplots_S3      3           # Generate few (1), some (3), or many (5) figures (Options: 1 - 5)
nplots          5           # How many of each type of figure do you want to make per file?
vmin            0.99        # Sets the vmin of the color bar for Figure 3101.
vmax            1.01        # Sets the vmax of the color bar for Figure 3101.
time_axis       'y'         # Determines whether the time axis in Figure 3101 is along the y-axis ('y') or the x-axis ('x')
hide_plots      True        # If True, plots will automatically be closed rather than popping up
save_output     True        # Save outputs for use in S4
save_fluxdata   False       # Save the much larger FluxData.h5 outputs which can be useful for debugging or comparisons between different pipelines
verbose         True        # If True, more details will be printed about steps

# Project directory
topdir                  /home/chetan/eureka/k2-18b/

# Directories relative to topdir
inputdir		        stage2
outputdir               stage3
"""

s3_path = ecf_dir / 'S3_k2-18b.ecf'
s3_path.write_text(s3_ecf_content)
print("S3_k2-18b.ecf written to:", s3_path)
print(s3_ecf_content)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Run Stage 3
# ─────────────────────────────────────────────────────────────────────────────

eventlabel = 'k2-18b'

ecf_path = '.'+os.sep

if __name__ == '__main__':
    # To skip one or more stages that were already run,
    # just comment them out below

    #meta = s1.rampfitJWST(eventlabel, ecf_path=ecf_path)

    #meta = s2.calibrateJWST(eventlabel, ecf_path=ecf_path)

    spec, meta = s3.reduce(eventlabel, ecf_path=ecf_path)

    #spec, lc, meta = s4.genlc(eventlabel, ecf_path=ecf_path)

    #meta = s5.fitlc(eventlabel, ecf_path=ecf_path)

    #meta, lc = s6.plot_spectra(eventlabel, ecf_path=ecf_path)

---
## 5. Stage 4 — Light Curve Construction

Stage 4 bins the per-integration 1D spectra into wavelength bins and produces spectrophotometric light curves. This is where you define:

- **Wavelength binning** — the width and positions of spectral channels
- **Time array construction** — converting BJD times to phases or hours from mid-transit
- **Stellar variability tracking** — saving the white-light (broad-band) light curve for detrending

### Wavelength bin strategy for K2-18b MIRI LRS

The optimal bin size balances:
- **SNR**: wider bins → more photons → lower noise
- **Spectral resolution**: narrower bins → more spectral information on molecular features
- **Systematics**: some systematic trends are wavelength-dependent


In [ ]:
s4_ecf_content = """\
# Eureka! Control File for Stage 4: Generate Lightcurves

# Stage 4 Documentation: https://eurekadocs.readthedocs.io/en/latest/ecf.html#stage-4

# Number of spectroscopic channels spread evenly over given wavelength range
nspecchan       29          # Number of spectroscopic channels spread evenly over given wavelength range. Set to None to leave the spectrum unbinned.
compute_white   True        # Also compute the white-light lightcurve
s4_order        None        # For NIRISS, specify spectral order
wave_min        5.24        # Minimum wavelength. Set to None to use the shortest extracted wavelength from Stage 3.
wave_max        12.02       # Maximum wavelength. Set to None to use the longest extracted wavelength from Stage 3.
allapers        False       # Run S4 on all of the apertures considered in S3? Otherwise will use newest output in the inputdir

# Manually mask pixel columns by index number
#mask_columns  []

# Parameters for drift correction of 1D spectra
recordDrift     True    # Set True to record drift/jitter in 1D spectra (always recorded if correctDrift is True)
correctDrift    False   # Set True to correct drift/jitter in 1D spectra (not recommended)
drift_preclip   0       # Ignore first drift_preclip points of spectrum
drift_postclip  100     # Ignore last drift_postclip points of spectrum, None = no clipping
drift_range     11      # Trim spectra by +/-X pixels to compute valid region of cross correlation
drift_hw        5       # Half-width in pixels used when fitting Gaussian, must be smaller than drift_range
drift_iref      -1      # Index of reference spectrum used for cross correlation, -1 = last spectrum
sub_mean        True    # Set True to subtract spectrum mean during cross correlation
sub_continuum   True    # Set True to subtract the continuum from the spectra using a highpass filter
highpassWidth   10      # The integer width of the highpass filter when subtracting the continuum

# Parameters for sigma clipping
clip_unbinned   False   # Whether or not sigma clipping should be performed on the unbinned 1D time series
clip_binned     True    # Whether or not sigma clipping should be performed on the binned 1D time series
sigma           5       # The number of sigmas a point must be from the rolling median to be considered an outlier
box_width       20      # The width of the box-car filter (used to calculated the rolling median) in units of number of data points
maxiters        20      # The number of iterations of sigma clipping that should be performed.
boundary        fill    # Use 'fill' to extend the boundary values by the median of all data points (recommended), 'wrap' to use a periodic boundary, or 'extend' to use the first/last data points
fill_value      mask    # Either the string 'mask' to mask the outlier values (recommended), 'boxcar' to replace data with the mean from the box-car filter, or a constant float-type fill value.

# Limb-darkening parameters
compute_ld      False   # Options: exotic-ld, spam, False
# Options for ExoTiC-LD
metallicity     0.12    # Metallicity of the star
teff            3464    # Effective temperature of the star in K
logg            4.7684     # Surface gravity in log g
exotic_ld_direc /home/rafael/hard_disk/exotic-ld_data/ # Directory for ancillary files for exotic-ld, download from: https://zenodo.org/doi/10.5281/zenodo.6047317
exotic_ld_grid  3D      # You can choose from kurucz (or 1D), stagger (or 3D), mps1, or mps2 model grids, or custom (to use custom_si_grid below)
# custom_si_grid  /home/User/path/to/custom/stellar/intensity/profile  #Custom Stellar Intensity profile. For examples see Eureka/demos/JWST/Custom_Stellar_Intensity_Files
# exotic_ld_file  /home/User/exotic-ld_data/Custom_throughput_file.csv # Custom throughput file, for examples see Eureka/demos/JWST/Custom_throughput_files

# Diagnostics
isplots_S4      5       # Generate few (1), some (3), or many (5) figures (Options: 1 - 5)
vmin            0.99    # Sets the vmin of the color bar for Figure 4101.
vmax            1.01    # Sets the vmax of the color bar for Figure 4101.
time_axis       'y'     # Determines whether the time axis in Figure 4101 is along the y-axis ('y') or the x-axis ('x')
hide_plots      True    # If True, plots will automatically be closed rather than popping up
verbose         True    # If True, more details will be printed about steps

# Project directory
topdir                  /home/chetan/eureka/k2-18b/

# Directories relative to topdir
inputdir		        stage3
outputdir               stage4
"""

s4_path = ecf_dir / 'S4_k2-18b.ecf'
s4_path.write_text(s4_ecf_content)
print("S4_k2-18b.ecf written to:", s4_path)
print(s4_ecf_content)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Run Stage 4
# ─────────────────────────────────────────────────────────────────────────────

eventlabel = 'k2-18b'

ecf_path = '.'+os.sep

if __name__ == '__main__':
    # To skip one or more stages that were already run,
    # just comment them out below

    #meta = s1.rampfitJWST(eventlabel, ecf_path=ecf_path)

    #meta = s2.calibrateJWST(eventlabel, ecf_path=ecf_path)

    #spec, meta = s3.reduce(eventlabel, ecf_path=ecf_path)

    spec, lc, meta = s4.genlc(eventlabel, ecf_path=ecf_path)

    #meta = s5.fitlc(eventlabel, ecf_path=ecf_path)

    #meta, lc = s6.plot_spectra(eventlabel, ecf_path=ecf_path)

---
## 6. Stage 5 — Light Curve Fitting

Stage 5 fits a transit model to each spectrophotometric light curve using either MCMC (via `emcee`) or nested sampling (via `dynesty`). The model is:

$$F(t) = T(t; R_p/R_\star, a/R_\star, i, u_1, u_2) \times S(t; \theta_{\rm sys})$$

where $T$ is the transit model (via `batman`) and $S$ is a systematics model.

### Transit model parameters for K2-18b

| Parameter | Prior type | Starting value | Notes |
|---|---|---|---|
| `rp` | free (uniform) | 0.05875 | Rp/R★ — the key science parameter |
| `t0` | free (normal) | 2457264.39116 | BJD_TDB, tight Gaussian |
| `inc` | free (uniform) | 89.5785 | degrees |
| `ars` | fixed | 72.77 | a/R★ |
| `e` | fixed | 0 | circular orbit |
| `w` | fixed | 90 | argument of periastron |
| `limb_dark` | quadratic | [u1, u2] | from ExoTiC-LD / Stage 4 |
| `scatter_mult` | free | 1.0 | error bar inflation |

### Systematics model

MIRI LRS exhibits a well-known **ramp** systematic at the start of each visit and possible **pointing jitter** correlated with the trace position. Eureka! supports:

- **Polynomial** trends in time (`polynomial_params`)
- **GP** (Gaussian Process) detrending via `celerite2`
- **Exponential ramp** correction: `s * exp(-τ * (t - t_start)) + c`

In [ ]:
s5_ecf_content = """\
# Eureka! Control File for Stage 5: Lightcurve Fitting

# Stage 5 Documentation: https://eurekadocs.readthedocs.io/en/latest/ecf.html#stage-5

ncpu             16

allapers         False

# NORMAL FITTING
fit_method       [emcee]
run_myfuncs      [batman_tr, polynomial, expramp, xpos, ypos, xwidth, ywidth]
fit_par          S5_k2-18b.epf

# Manual clipping in time
manual_clip      [[None,250]]

# Limb darkening controls
use_generate_ld  None
ld_file          None
ld_file_white    None
recenter_ld_prior True

# PyMC3 NUTS sampler settings
#tune             3000
#draws            1000
#chains           3
#target_accept    0.85

#mcmc
old_chain       None     # Output folder relative to topdir that contains an old emcee chain to resume where you left off (set to None to start from scratch)
lsq_first       False    # Initialize with an initial lsq call (can help shorten burn-in, but turn off if lsq fails). Only used if old_chain is None
run_nsteps      3000
run_nwalkers    200
run_nburn       1500     # How many of run_nsteps should be discarded as burn-in steps

#dynesty
run_nlive       121  
run_bound       multi
run_sample      rwalk
run_tol         0.01

# Plotting controls
interp           False

# Diagnostics
isplots_S5       5
nbin_plot        None
hide_plots       True
verbose          True

# Project directory
topdir                  /home/chetan/eureka/k2-18b/

# Directories relative to topdir
inputdir		        stage4
outputdir               stage5
"""

s5_path = ecf_dir / 'S5_k2-18b.ecf'
s5_path.write_text(s5_ecf_content)
print("S5_k2-18b.ecf written to:", s5_path)
print(s5_ecf_content)

In [ ]:
s5_epf_content = """\

# Name      Value           Free?			    PriorPar1      PriorPar2  PriorType
# Free? can be free, fixed, white_free, white_fixed, shared, or independent
# PriorType can be U (Uniform), LU (Log Uniform), or N (Normal).
# If U/LU, PriorPar1 and PriorPar2 represent upper and lower limits of the parameter/log(the parameter).
# If N, PriorPar1 is the mean and PriorPar2 is the standard deviation of a Gaussian prior.
#-------------------------------------------------------------------------------------------------------
# ------------------
# ** Transit/eclipse parameters **
# ------------------
rp          0.05375         'free'        0.05375         0.04     N
# ------------------
# Orbital parameters
# ------------------
per         32.940045       'fixed'
t0          60426.12874820823        'free'        60426.12        0.1      N
time_offset 0               'independent'
inc         89.55526413525133          'free'        89.0            2.0      N
a           79.98620841069588          'free'        80.3            2.0      N
ecc         0.0             'fixed'
w           90.             'fixed'
# -------------------------
# Limb darkening parameters
# Choose limb_dark from ['uniform', 'linear', 'quadratic', 'kipping2013', 'square-root', 'logarithmic', 'exponential', '4-parameter']
# -------------------------
##### ExoTiC-LD #####
limb_dark   'kipping2013'     'independent'
u1          0.037           'free'        0.03            0.1      N
u2          0.114           'free'        0.11            0.1      N
# --------------------
# Systematic variables
# --------------------
c0          1.001           'free'        1.001           0.1     N
c1          -0.001          'free'        -0.001          0.1     N
### c2          -0.001          'free'        -0.001          0.1     N
r0          0.0             'free'       0          0.1        N
r1          10.0            'free'      0.1        200      U
# Ramp model parameters
# Centroid decorrelation parameters
### ypos        -0.01           'free'        0               0.1      N
### xpos        0.09            'free'        0               0.1      N
# -----------
# White noise
# -----------
scatter_mult 1.44           'free'        0.8             4        U
"""

s5_path = ecf_dir / 'S5_k2-18b.epf'
s5_path.write_text(s5_epf_content)
print("S5_k2-18b.epf written to:", s5_path)
print(s5_epf_content)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Run Stage 5
# ─────────────────────────────────────────────────────────────────────────────

eventlabel = 'k2-18b'

ecf_path = '.'+os.sep

if __name__ == '__main__':
    # To skip one or more stages that were already run,
    # just comment them out below

    #meta = s1.rampfitJWST(eventlabel, ecf_path=ecf_path)

    #meta = s2.calibrateJWST(eventlabel, ecf_path=ecf_path)

    #spec, meta = s3.reduce(eventlabel, ecf_path=ecf_path)

    #spec, lc, meta = s4.genlc(eventlabel, ecf_path=ecf_path)

    meta = s5.fitlc(eventlabel, ecf_path=ecf_path)

    #meta, lc = s6.plot_spectra(eventlabel, ecf_path=ecf_path)

---
## 7. Stage 6 — Transmission Spectrum & Visualisation

Stage 6 collects the per-channel $(R_p/R_\star)^2$ values from Stage 5 and assembles them into the **transmission spectrum** — the primary science deliverable. 

The transmission spectrum encodes information about the planetary atmosphere:

$$\delta(\lambda) = \left(\frac{R_p}{R_\star}\right)^2 = \frac{(R_0 + h_{\rm eff}(\lambda))^2}{R_\star^2}$$

where $h_{\rm eff}(\lambda)$ is the effective atmospheric height, which is large at wavelengths where atmospheric gases absorb and small in windows of transparency.

In [ ]:
s6_ecf_content = """\

# Eureka! Control File for Stage 6: Spectra Plotting

allapers        False  # Run S6 on all of the apertures considered in S5? Otherwise will use newest output in the inputdir

# Plotting parameters
y_params        ['rp', 'rp^2', 'u1', 'u2', 'c0', 'c1', 'r0', 'r1', 'scatter_mult'] # The parameter name as entered in your EPF file in Stage 5
y_labels        None   # The formatted string you want on the y-label
y_label_units   None   # The formatted string for the units you want on the y-label - e.g., (ppm), (seconds), '(days$^-1)$', etc.. Set to None to automatically add any implied units from y_scalars (e.g. ppm), or set to '' to force no units.
y_scalars       [1, 100, 1, 1, 1, 1, 1, 1, 1]   # Can be used to convert to percent (100), ppm (1e6), etc.
x_unit          um     # Options include any measurement of light included in astropy.units.spectral (e.g. um, nm, Hz, etc.)

# Tabulating parameters
ncol            4      # The number of columns you want in your LaTeX formatted tables

# This section is relevant if isplots_S6>=3 and plotting rp or rp^2.
# Scale height parameters (if you want a second copy of the plot with a second y-axis with units of scale height)
# -------------------
# K2-18 parameters
# -------------------
star_Rad        0.4445	# The radius of the star in units of solar radii
planet_Teq      255		# The equilibrium temperature of the planet in units of Kelvin
planet_Mass     0.0272	# The planet's mass in units of Jupiter radii (used to calculate surface gravity)
planet_Rad      None	# The planet's radius in units of Jupiter radii (used to calculate surface gravity); Set to None to use the average fitted radius
planet_mu       2.3		# The mean molecular mass of the atmosphere (in atomic mass units)
planet_R0       None	# The reference radius (in Jupiter radii) for the scale height measurement; Set to None to use the mean fitted radius

# Diagnostics
isplots_S6      5      # Generate few (1), some (3), or many (5) figures (Options: 1 - 5)
testing_S6      False
hide_plots      True    # If True, plots will automatically be closed rather than popping up

# Project directory
topdir                  /home/chetan/eureka/k2-18b/

# Directories relative to topdir
inputdir		        stage5
outputdir               stage6
"""

s6_path = ecf_dir / 'S6_k2-18b.ecf'
s6_path.write_text(s6_ecf_content)
print("S6_k2-18b.ecf written to:", s6_path)
print(s6_ecf_content)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Run Stage 6
# ─────────────────────────────────────────────────────────────────────────────

eventlabel = 'k2-18b'

ecf_path = '.'+os.sep

if __name__ == '__main__':
    # To skip one or more stages that were already run,
    # just comment them out below

    #meta = s1.rampfitJWST(eventlabel, ecf_path=ecf_path)

    #meta = s2.calibrateJWST(eventlabel, ecf_path=ecf_path)

    #spec, meta = s3.reduce(eventlabel, ecf_path=ecf_path)

    #spec, lc, meta = s4.genlc(eventlabel, ecf_path=ecf_path)

    #meta = s5.fitlc(eventlabel, ecf_path=ecf_path)

    meta, lc = s6.plot_spectra(eventlabel, ecf_path=ecf_path)